# football-laya — curva de aprendizagem no Kaggle (T4 grátis)

Treina a grelha 2000 / 5000 / 10000 / 16019 (todos) — casos mais recentes — + avalia com calibração no test fixo.
Matemática **idêntica** à local (mesmos scripts, mesmos hiperparams) — os números são comparáveis.

**Antes de correr:** Settings do notebook → Accelerator **GPU T4 x2** (usamos 1) → **Internet ON** (pip).
**Limites:** sessões até ~12h, quota ~30h GPU/semana. O treino faz `resume.pt` de 200 em 200 steps:
se a sessão cair, corre a célula de treino outra vez e continua sozinho.

In [ ]:
!nvidia-smi -L
import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
!pip install -q laya==0.3.26

In [ ]:
import shutil, zipfile
from pathlib import Path

roots = [Path('/kaggle/input')]
zips = list(Path('/kaggle/input').rglob('*.zip'))
if zips:  # dataset subido com --dir-mode zip: extrair primeiro
    tmp = Path('/tmp/ds')
    for z in zips:
        with zipfile.ZipFile(z) as f:
            f.extractall(tmp / z.stem)
    roots.append(tmp)

def find_dir(marker):
    for r in roots:
        hits = list(r.rglob(marker))
        if hits:
            return hits[0].parent
    raise AssertionError(marker + ' nao encontrado: liga o dataset em Add input!')

DST = Path('/kaggle/working/fb')
for marker, dest in (('processed/train.jsonl', DST / 'data' / 'processed'),
                     ('rl_agent_config.json', DST / 'data' / 'models' / 'laya-multilingual-base'),
                     ('scripts/train_curve.py', DST / 'scripts')):
    src = find_dir(marker)
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
print('tree pronta')


In [ ]:
# smoke test: tokeniza 100 casos, sem treinar (~1-2 min)
!python /kaggle/working/fb/scripts/train_curve.py --checkpoints 100 --dry-run --curve-dir /kaggle/working/fb/data/models/curve_kaggle

In [ ]:
# TREINO (horas; pode interromper — resume sozinho ao re-correr esta celula)
!python /kaggle/working/fb/scripts/train_curve.py --checkpoints 2000,5000,10000,16019 --batch 32 --accum 1 --amp --grad-checkpoint --save-every 200 --curve-dir /kaggle/working/fb/data/models/curve_kaggle

In [ ]:
# AVALIACAO (calibra no calib fixo, mede no test fixo; salta quem ja tem metrics)
!python /kaggle/working/fb/scripts/eval_curve.py --checkpoints 0,2000,5000,10000,16019 --batch 128 --curve-dir /kaggle/working/fb/data/models/curve_kaggle --metrics-dir /kaggle/working/fb/data/processed/curve_kaggle

In [ ]:
!column -s, -t /kaggle/working/fb/data/processed/curve_kaggle/metrics.csv 2>/dev/null || cat /kaggle/working/fb/data/processed/curve_kaggle/metrics.csv
!ls /kaggle/working/fb/data/models/curve_kaggle/

## Notas
- **Resume:** sessão caiu? Corre as células 4→6 outra vez. O treino retoma do `resume.pt`, o eval salta checkpoints prontos.
- **Guardar resultados:** Save Version (output) ou download manual de `data/models/curve_kaggle/n_*/model.safetensors` + `data/processed/curve_kaggle/metrics.csv`.
- **Comparar com local:** o `metrics.csv` tem as mesmas colunas do piloto — acc_noul_cal, ece_noul_cal, noul_conf_cal por checkpoint.
- **Quota:** ~30h/semana (recomeça sábado). Esta grelha ≈ 25h — se acabar a meio, continua no sábado seguinte (o resume guarda tudo).